Importing the dataset

In [1]:
import pandas as pd

food = pd.read_csv("food.csv")
food_nutrient = pd.read_csv("food_nutrient.csv")
nutrient = pd.read_csv("nutrient.csv")

food.head()


C:\Users\Vandan Agrawal\AppData\Local\Temp\ipykernel_3368\763829690.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("food_nutrient.csv")


,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01


Just keeping raw ingredients and removing store bought packaged foods for more accuracy

In [2]:
food_filtered = food[
    food["description"].str.contains(
        "raw|fresh|oil|flour|cheese|milk|egg|tomato|onion|garlic",
        case=False,
        na=False
    )
]


In [3]:
food_filtered["description"].sample(20)


28365           SOY MILK, UNSWEETENED, PLAIN, SHELF STABLE
68083            Sorghum bran, white, unenriched, dry, raw
21524        ALMOND MILK, UNSWEETENED, PLAIN, SHELF STABLE
9796     Vitamin B6, Ground turkey, 93% lean, 7% fat, p...
31824                         Flour, Semolina, Semi-Coarse
37172                              peppers, bell, red, raw
50944                                        cassava flour
36012               almond milk, refrigerated, unsweetened
22552           SOY MILK, UNSWEETENED, PLAIN, SHELF STABLE
52000                                        quinoa, flour
6429     Cantaloupe, Raw, Pass 2, Region 4, n/a, Yes, R...
24650                                          OIL, PEANUT
26576                              OIL, OLIVE, EXTRA LIGHT
43016                              blueberries, fresh, raw
52749                                        barley, flour
64935              Plantains, black, overripe, peeled, raw
28864        ALMOND MILK, UNSWEETENED, PLAIN, SHELF STAB

Building ingredient table list 

In [4]:
# STEP B: Build ingredient lookup table from USDA FDC Foundation Foods
# Produces: ingredient_lookup.csv with fdc_id -> canonical ingredient
# Also includes the original FDC description for sanity checks.

import re
import pandas as pd

# -----------------------------
# 1) Load FDC files (Foundation Foods)
# -----------------------------
FOOD_CSV_PATH = "food.csv"  # adjust path if needed
food = pd.read_csv(FOOD_CSV_PATH)

# Safety: keep only columns we need (avoid huge memory use)
needed_cols = [c for c in ["fdc_id", "description", "data_type", "food_category_id"] if c in food.columns]
food = food[needed_cols].copy()

# -----------------------------
# 2) Quick filter to reduce obviously non-ingredient entries
#    (Foundation Foods is already ingredient-focused, but still has some prepared items)
# -----------------------------
# You can relax/tighten this later. For now: keep rows with a description.
food = food.dropna(subset=["description"]).copy()
food["description"] = food["description"].astype(str)

# Optional: if data_type exists, keep Foundation only
if "data_type" in food.columns:
    # Foundation foods dataset typically uses "foundation_food"
    food = food[food["data_type"].str.contains("foundation", case=False, na=False)].copy()

# -----------------------------
# 3) Canonical ingredient mapping (expandable)
#    Order matters: more specific patterns should go first.
# -----------------------------
PATTERN_TO_CANONICAL = [
    # Oils & fats
    (r"\bolive oil\b|\boil, olive\b", "olive_oil"),
    (r"\bcanola oil\b", "canola_oil"),
    (r"\bvegetable oil\b", "vegetable_oil"),
    (r"\bbutter\b", "butter"),

    # Core Italian vegetables
    (r"\btomato(es)?\b", "tomato"),
    (r"\bonion(s)?\b", "onion"),
    (r"\bgarlic\b", "garlic"),
    (r"\bbasil\b", "basil"),
    (r"\boregano\b", "oregano"),
    (r"\bparsley\b", "parsley"),
    (r"\brosemary\b", "rosemary"),
    (r"\bthyme\b", "thyme"),
    (r"\bspinach\b", "spinach"),
    (r"\bmushroom(s)?\b", "mushroom"),
    (r"\bzucchini\b", "zucchini"),
    (r"\beggplant\b|\baubergine\b", "eggplant"),
    (r"\bpepper\b", "pepper"),
    (r"\bbroccoli\b", "broccoli"),
    (r"\bkale\b", "kale"),

    # Fruits commonly used
    (r"\blemon\b", "lemon"),
    (r"\bbanana(s)?\b", "banana"),
    (r"\bblueberr(y|ies)\b", "blueberry"),
    (r"\bpeach(es)?\b", "peach"),
    (r"\bavocado\b", "avocado"),
    (r"\bapricot(s)?\b", "apricot"),
    (r"\bkiwi(fruit)?\b", "kiwi"),
    (r"\bcantaloupe\b|\bmelon\b", "melon"),

    # Beans & legumes
    (r"\bbeans?\b", "beans"),
    (r"\bkidney beans?\b", "kidney_beans"),
    (r"\bblack beans?\b", "black_beans"),
    (r"\bpinto beans?\b", "pinto_beans"),
    (r"\blentils?\b", "lentils"),
    (r"\bchickpeas?\b|\bgarbanzo\b", "chickpeas"),

    # Grains & pasta
    (r"\bpasta\b|\bspaghetti\b|\bpenne\b|\bfusilli\b|\bmacaroni\b", "pasta"),
    (r"\brace\b", "rice"),
    (r"\bflour\b", "flour"),

    # Dairy
    (r"\bparmesan\b|\bparmigiano\b", "parmesan_cheese"),
    (r"\bmozzarella\b", "mozzarella"),
    (r"\bricotta\b", "ricotta"),
    (r"\bpecorino\b", "pecorino"),
    (r"\bmilk\b", "milk"),
    (r"\byogurt\b", "yogurt"),

    # Protein sources
    (r"\begg(s)?\b", "egg"),
    (r"\bchicken\b", "chicken"),
    (r"\bbeef\b", "beef"),
    (r"\bpork\b", "pork"),
    (r"\bsalmon\b", "salmon"),
    (r"\btuna\b", "tuna"),
    (r"\bsnapper\b", "snapper"),
    (r"\bswordfish\b", "swordfish"),
    (r"\bbison\b", "bison"),

    # Nuts
    (r"\bpecans?\b", "pecans"),
    (r"\bpine nuts?\b", "pine_nuts"),
    (r"\bwalnuts?\b", "walnuts"),
    (r"\balmonds?\b", "almonds"),

    # Pantry basics
    (r"\bsalt\b", "salt"),
    (r"\bsugar(s)?\b", "sugar"),
]


# Precompile regexes for speed
COMPILED = [(re.compile(pat, flags=re.IGNORECASE), canon) for pat, canon in PATTERN_TO_CANONICAL]

def normalize_ingredient(desc: str) -> str | None:
    """Map an FDC description string to a canonical ingredient name."""
    if not isinstance(desc, str) or not desc.strip():
        return None

    d = desc.lower()

    for rx, canon in COMPILED:
        if rx.search(d):
            return canon

    return None  # unknown/unmapped

# -----------------------------
# 4) Apply mapping
# -----------------------------
food["ingredient"] = food["description"].apply(normalize_ingredient)

# Keep both mapped and unmapped so you can inspect what you're missing
mapped = food.dropna(subset=["ingredient"]).copy()
unmapped = food[food["ingredient"].isna()].copy()

# -----------------------------
# 5) Write outputs
# -----------------------------
# Main lookup (mapped only)
ingredient_lookup = mapped[["fdc_id", "ingredient", "description"]].drop_duplicates()
ingredient_lookup.to_csv("ingredient_lookup.csv", index=False)

# Helpful debug file: what didn't map
unmapped[["fdc_id", "description"]].drop_duplicates().to_csv("ingredient_unmapped_debug.csv", index=False)

# Summary print
print("✅ ingredient_lookup.csv saved")
print("Rows (mapped):", ingredient_lookup.shape[0])
print("Unique ingredients:", ingredient_lookup["ingredient"].nunique())
print("\nTop canonical ingredients:")
print(ingredient_lookup["ingredient"].value_counts().head(20))

print("\n✅ ingredient_unmapped_debug.csv saved")
print("Rows (unmapped):", unmapped.shape[0])


✅ ingredient_lookup.csv saved
Rows (mapped): 243
Unique ingredients: 42

Top canonical ingredients:
ingredient
beans        47
flour        30
beef         24
milk         18
mushroom     15
pork         13
tomato       10
chicken       9
egg           9
butter        6
banana        5
onion         5
spinach       4
sugar         3
salt          3
olive_oil     3
yogurt        3
broccoli      2
kale          2
ricotta       2
Name: count, dtype: int64

✅ ingredient_unmapped_debug.csv saved
Rows (unmapped): 193


Attaching nutrient values to the ingredients

In [5]:
import pandas as pd

# -----------------------------
# Load files
# -----------------------------
ingredient_lookup = pd.read_csv("ingredient_lookup.csv")
food_nutrient = pd.read_csv("food_nutrient.csv")
nutrient = pd.read_csv("nutrient.csv")

# -----------------------------
# Keep nutrients we need
# -----------------------------
wanted_nutrients = {
    "Energy": "calories",
    "Protein": "protein",
    "Total lipid (fat)": "fat",
    "Carbohydrate, by difference": "carbs"
}

nutrient_subset = nutrient[
    nutrient["name"].isin(wanted_nutrients.keys())
][["id", "name"]]

nutrient_subset["nutrient_name"] = nutrient_subset["name"].map(
    wanted_nutrients
)

# -----------------------------
# Merge nutrient values
# -----------------------------
merged = food_nutrient.merge(
    nutrient_subset,
    left_on="nutrient_id",
    right_on="id"
)

merged = merged.merge(
    ingredient_lookup,
    on="fdc_id"
)

# -----------------------------
# Keep needed columns
# -----------------------------
ingredient_nutrients = merged[
    ["ingredient", "nutrient_name", "amount"]
]

# -----------------------------
# Aggregate per ingredient
# -----------------------------
ingredient_nutrients = (
    ingredient_nutrients
    .groupby(["ingredient", "nutrient_name"])
    ["amount"]
    .mean()
    .reset_index()
)

# -----------------------------
# Save table
# -----------------------------
ingredient_nutrients.to_csv(
    "ingredient_nutrient_table.csv",
    index=False
)

print("✅ Ingredient nutrient table created")
print(ingredient_nutrients.head())


✅ Ingredient nutrient table created
  ingredient nutrient_name      amount
0    almonds      calories  1605.00000
1    almonds         carbs    18.11731
2    almonds           fat    54.44500
3    almonds       protein    20.92519
4    apricot         carbs    10.23875


C:\Users\Vandan Agrawal\AppData\Local\Temp\ipykernel_3368\2235284763.py:7: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("food_nutrient.csv")


Actually creating the FOPCs

In [6]:
import pandas as pd
from pathlib import Path

INPUT_CSV = "ingredient_nutrient_table.csv"

OUT_DIR = Path(".")
ING_FACTS_PATH = OUT_DIR / "facts_ingredients.pl"
NUTR_FACTS_PATH = OUT_DIR / "facts_nutrients.pl"

def safe_atom(s: str) -> str:
    s = str(s).strip().lower()
    cleaned = []
    for ch in s:
        if ch.isalnum():
            cleaned.append(ch)
        else:
            cleaned.append("_")
    s = "".join(cleaned)

    while "__" in s:
        s = s.replace("__", "_")

    s = s.strip("_")
    if not s:
        return "unknown"
    if s[0].isdigit():
        s = "n_" + s
    return s

def format_float(x) -> str:
    try:
        fx = float(x)
    except Exception:
        return "0"
    return f"{fx:.6g}"

df = pd.read_csv(INPUT_CSV)

df["ingredient_atom"] = df["ingredient"].apply(safe_atom)
df["nutrient_atom"] = df["nutrient_name"].apply(safe_atom)

unique_ings = sorted(df["ingredient_atom"].unique())

with open(ING_FACTS_PATH, "w", encoding="utf-8") as f:
    f.write("% Ingredient facts\n")
    for ing in unique_ings:
        f.write(f"Ingredient({ing}).\n")

with open(NUTR_FACTS_PATH, "w", encoding="utf-8") as f:
    f.write("% Nutrient facts per 100g\n")
    for _, row in df.iterrows():
        ing = row["ingredient_atom"]
        nutr = row["nutrient_atom"]
        amt = format_float(row["amount"])
        f.write(f"HasNutrient({ing}, {nutr}).\n")
        f.write(f"NutrientValuePer100g({ing}, {nutr}, {amt}).\n")

print("Facts generated successfully!")


Facts generated successfully!


In [7]:
from pyDatalog import pyDatalog
import re
from pathlib import Path

pyDatalog.create_terms("""
    Ingredient, HasNutrient, NutrientValuePer100g,
    I, N, V
""")

def load_facts(file_path):
    fact_re = re.compile(r"^\s*([A-Za-z_]\w*)\((.*)\)\s*\.\s*$")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%"):
                continue

            m = fact_re.match(line)
            if not m:
                continue

            pred = m.group(1)
            args = [a.strip() for a in m.group(2).split(",")]

            parsed_args = []
            for a in args:
                try:
                    parsed_args.append(float(a))
                except:
                    parsed_args.append(a)

            if pred == "Ingredient":
                +Ingredient(parsed_args[0])
            elif pred == "HasNutrient":
                +HasNutrient(parsed_args[0], parsed_args[1])
            elif pred == "NutrientValuePer100g":
                +NutrientValuePer100g(parsed_args[0], parsed_args[1], parsed_args[2])

load_facts("facts_ingredients.pl")
load_facts("facts_nutrients.pl")

print("Facts loaded!")

# Example query
ingredient = "tomato"
print("\nNutrients in tomato:")
print(HasNutrient(ingredient, N))

print("\nCalories per 100g tomato:")
print(NutrientValuePer100g(ingredient, "calories", V))


Facts loaded!

Nutrients in tomato:
N       
--------
fat     
protein 
carbs   
calories

Calories per 100g tomato:
V   
----
58.5


In [8]:
import pandas as pd
import re

# ---- Paths (edit if needed) ----
ING_LOOKUP_PATH = "ingredient_lookup.csv"
OUT_SELECTED_PATH = "ingredient_lookup_selected.csv"

df = pd.read_csv(ING_LOOKUP_PATH)

NEGATIVE_TERMS = [
    "canned", "cooked", "fried", "roasted", "grilled", "baked",
    "frozen", "prepared", "with", "added", "sweetened", "in sauce",
    "pickled", "smoked", "breaded", "seasoned", "dried", "dehydrated"
]

POSITIVE_TERMS = [
    "raw", "fresh"
]

# Optional ingredient-specific boosts (helps pick a more “standard” entry)
PREFERRED_REGEX = {
    "tomato": [r"tomatoes, red, ripe, raw", r"tomato.*raw"],
    "onion":  [r"onions?, raw"],
    "garlic": [r"garlic.*raw"],
    "olive_oil": [r"oil, olive(,|$)"],
}

def score_description(ingredient: str, desc: str) -> float:
    d = str(desc).lower()
    score = 0.0

    # Positive boosts
    for t in POSITIVE_TERMS:
        if t in d:
            score += 5.0 if t == "raw" else 2.0

    # Negative penalties
    for t in NEGATIVE_TERMS:
        if t in d:
            score -= 4.0

    # Ingredient-specific preferences
    if ingredient in PREFERRED_REGEX:
        for rx in PREFERRED_REGEX[ingredient]:
            if re.search(rx, d):
                score += 10.0

    # Tie-breaker: shorter descriptions usually more “basic”
    score += max(0, 3.0 - (len(d) / 80.0))  # small bonus for shorter

    return score

df["score"] = df.apply(lambda r: score_description(r["ingredient"], r["description"]), axis=1)

# Pick best row per canonical ingredient
selected = (
    df.sort_values(["ingredient", "score"], ascending=[True, False])
      .groupby("ingredient", as_index=False)
      .head(1)
      .reset_index(drop=True)
)

selected.to_csv(OUT_SELECTED_PATH, index=False)

print("✅ Saved:", OUT_SELECTED_PATH)
print("Unique ingredients:", selected["ingredient"].nunique())
print("\nSample selections:")
print(selected[["ingredient", "fdc_id", "description", "score"]].head(15))


✅ Saved: ingredient_lookup_selected.csv
Unique ingredients: 42

Sample selections:
   ingredient   fdc_id                                        description  \
0     almonds  2346393                          Nuts, almonds, whole, raw   
1     apricot  2710815                            Apricot, with skin, raw   
2     avocado  2710824                         Avocado, Hass, peeled, raw   
3      banana   790774                             Bananas, overripe, raw   
4       beans  2346400                            Beans, snap, green, raw   
5        beef  2727573                        Beef, tenderloin steak, raw   
6       bison  2727571                                 Bison, ground, raw   
7   blueberry  2263889                                   Blueberries, raw   
8    broccoli   321900                                      Broccoli, raw   
9      butter   790508                              Butter, stick, salted   
10    chicken  2727568                  Chicken, wing, meat and skin, 

In [9]:
def print_fopc_examples(file_path, n=15):
    print(f"\n--- Sample facts from {file_path} ---\n")
    
    with open(file_path, "r", encoding="utf-8") as f:
        count = 0
        for line in f:
            if line.startswith("%"):
                continue
            print(line.strip())
            count += 1
            if count >= n:
                break

print_fopc_examples("facts_ingredients.pl", 20)
print_fopc_examples("facts_nutrients.pl", 20)



--- Sample facts from facts_ingredients.pl ---

Ingredient(almonds).
Ingredient(apricot).
Ingredient(avocado).
Ingredient(banana).
Ingredient(beans).
Ingredient(beef).
Ingredient(bison).
Ingredient(blueberry).
Ingredient(broccoli).
Ingredient(butter).
Ingredient(chicken).
Ingredient(egg).
Ingredient(eggplant).
Ingredient(flour).
Ingredient(garlic).
Ingredient(kale).
Ingredient(kiwi).
Ingredient(lentils).
Ingredient(melon).
Ingredient(milk).

--- Sample facts from facts_nutrients.pl ---

HasNutrient(almonds, calories).
NutrientValuePer100g(almonds, calories, 1605).
HasNutrient(almonds, carbs).
NutrientValuePer100g(almonds, carbs, 18.1173).
HasNutrient(almonds, fat).
NutrientValuePer100g(almonds, fat, 54.445).
HasNutrient(almonds, protein).
NutrientValuePer100g(almonds, protein, 20.9252).
HasNutrient(apricot, carbs).
NutrientValuePer100g(apricot, carbs, 10.2387).
HasNutrient(apricot, fat).
NutrientValuePer100g(apricot, fat, 0.405).
HasNutrient(apricot, protein).
NutrientValuePer100g(apr